# 08 - Simple text analysis

Original content from Felipe Alvarez de Toledo available at https://github.com/FelipeAdeT/PythonforHumanities

Content adaptation and modifications by Maroussia Bednarkiewicz.

### Navigation reminder

- **Grey cells** are **code cells**. Click inside them and type to edit.
- **Run**  code cells by pressing $ \triangleright $  in the toolbar above, or press ``` shift + enter```.
-  **Stop** a running process by clicking &#9634; in the toolbar above.
- You can **add new cells** by clicking to the left of a cell and pressing ```A``` (for above), or ```B``` (for below).
- **Delete cells** by pressing ```X``` or the bin symbol in the right upper corner of the cell.
- Run all code cells that import objects (such as the one below) to ensure that you can follow exercises and examples.
- Feel free to edit and experiment - you will not corrupt the original files.

## Introduction

In lessons 1-7, we learned about many of the building blocks that make Python a powerful language, including basic data types (strings, integers, floats, lists and dictionaries), conditional structures, and loops.

In this notebook, we will run through a long-form exercise that allows us to put all this knowledge together while simulating the typical process for a project using computation in the humanities. We will be working with a corpus of fairy tales by the brothers Grimm. In the first part of the notebook, we will **pre-process** or **clean** the text for text analysis. Then, we conduct an **exploratory analysis** of the corpus to become familiarized with the text and identify potential research questions. In the second part, we take a research question and perform a simple text analysis that uses the concepts we have learned in previous lessons.

There are several modules for text analysis that provide advanced tools for text analysis. In this lesson, however, we will use the basic building blocks of the Python language to construct simple tools for term counting and similarity analysis.

Please note that there are multiple paths to the objectives we outline below. We do provide a 'solution' notebook with our approach towards this project as guidance if you get stuck, but if you find alternative solutions, it is more useful to think critically about your code and whether it achieves its goals than to try to make it conform to the sample solution.

---

**Lesson Objectives**
- Practice:
    - Using basic data structures (strings and numbers) and their operators
    - Creating, populating and retrieving information from collections (lists and dictionaries)
    - Discerning when to choose one type of collection or another
    - Creating loops
    - Using conditional statements
- Develop good habits for projects, including:
    - Taking time to understand the source data and its particularities before committing to an approach
    - Thinking critically about algorithms and problem-solving, instead of immediately delegating to solutions developed by others
---

In [ ]:
# @title Grant GoogleColab access to your GoogleDrive and import questions for this notebook
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add your module folder to Python path
import sys
module_path = f"/content/drive/My Drive/IDH/Notebooks"
sys.path.append(module_path)
print("GoogleColab can now access your GoogleDrive.")

# 1. Load the text

The file we will be using is saved in 'Other_files/GrimmsFairyTales.txt'.  Using the 'with xxx as file' notation, create a file handle using the open() statement and then read the file as one block of text, assigning it to the variable text. This ensures that the file is closed after we read into it.

In [ ]:
# Import the necessary library
import os

In [ ]:
with open(os.path.join(f"{module_path}/Other_files", 'GrimmsFairyTales.txt'), 'r', encoding='utf-8') as file:
    text = file.read()

Text taken from the [Project Gutenberg](http://www.gutenberg.org/files/5314/5314-0.txt)

# 2. Examine a sample

Take a first look at the text file by printing characters 7,000 to 8,000 of the text file.

**Hint:** the object is like a long string, so you can index into it like as you usually would a string.

In [ ]:
print(text[7000:8000])

**3.** It is also helpful to display the text without formatting, so we can see invisible characters such as spaces or new lines (otherwise known as **whitespace**). Index into the whole text as before, but without using the print statement.

Start thinking about the **characteristics of the file**. How are the stories separated and structured?  How might we use whitespace to divide it into individual stories, and these stories into terms (also known as 'tokens')? What actions might we have to perform in order to extract these tokens for counting?

In [ ]:
text[7000:8000]

**We have observed these characteristics of the text that we see will affect our approach.**

- Stories are separated by four instances of the newline character ```\n``` (one to end the final line of the poem, and two to generate blank lines after).
- Words are separated by spaces or one or more instances the newline characte` ```\n``` (between the last word of a line and the first of the next).
- Some words begin with a capital letter, others are fully lowercase.
- Some words have punctuation around them.
    
We can use the first two characteristics to split the block of text into stories and further into terms or tokens. The other characteristics will have to be edited out to get a clean count of the tokens (to ensure, for instance, that 'Now' and 'now' or 'now.' and 'now' are counted as the same token).

# 3. Text pre-processing

How we choose to clean a text depends on our project's goal. We want to analyze term frequencies, meaning that our goal is to separate the text into discrete terms. The volume is a collection of stories, so it would also be useful to separate the text into story units.

To end up with a set of clean tokens, we will have to pre-process the text in several phases:
1. Strip the text of punctuation
1. Strip the text of cases (capitalization).
1. Split the text into tokens using whitespace.

Note that text cleaning could actually be undertaken at any point: we choose to do it first for the sake of efficiency, applying the changes to the whole text and avoiding creating loops that would otherwise have to re-iterate the step for each poem or for each token.

## Removing punctuation and cases

In this case, we want to very simply count term frequency, so we make the decision to remove capitals and any punctuation outside of tokens. This way, words will be counted as the same term regardless of how they were capitalized or if they had any adjacent punctuation marks.

In the code cell below, we create a string object that contains common punctuation marks.

In [ ]:
punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~’‘“”'

Recall that strings can be thought of as a series of characters, and a for-loop will iterate through each character in a string if used in the for-statement.

1. Using variable assignment and a string method, create a variable called 'text_clean' which is an all-lowercase version of the text variable.
1. Create a loop that iterates through the punctuation marks above and uses another string method to replace the marks from the clean_text with no character (this could be written as a string with nothing in between the quotes, or ''). Remember that string methods do not act in place, so you will have to reassign the clean_text variable within the loop.

In [ ]:
text_clean = text.lower()
for mark in punctuation:
    text_clean = text_clean.replace(mark,'')

In [ ]:
print(text_clean[7000:8000])

# 4. Exploratory analysis: finding the most common words in the corpus.

Before analyzing individual stories, let's have a look at the overall frequency of terms in the collection of Grimms Fairy Tales as a whole. This will allow us to view larger trends and think of potential research questions. To achieve this, we have to split the corpus into a collection of tokens.

> **Token**: An individual unit of meaning (most frequently, a word).

We will use the word 'token' interchangeably with 'term' or 'word' throughout this exercise.

> **Tokenization**:  The process of breaking down text document apart into tokens.

One issue we noticed before is that some words are separated by spaces, but others might be separated by newline characters (\n). Since we don't care about story structure in this exercise, the simplest solution is to replace line break characters with spaces. This will allow us to use the string.split() method afterwards using spaces to divide the stories up into tokens.

First, create a list called corpus_tokens by using two string methods: one to replace any line break characters in text_clean and one to then split the file by spaces. Do not overwrite text_clean, as we will be using it later to divide the text by stories.

In [ ]:
corpus_tokens = text_clean.replace('\n',' ').split()

In [ ]:
print(corpus_tokens[:100])

Now that the text is split into tokens, we can count the number of times each word appears in the text.

# Solving the problem: calculating term counts

We should think about the nature of our problem.

We have a data structure which is a list of tokens. As we iterate through the list, we will have to identify each distinct term, and then count the number of times it appears in the list. In other words, we will have a number of **unique terms** and an associated count **value** for each. This should ring some alarm bells about the best data structure for storing the information. What data structure would you use?

Ideally, we would want to create a dictionary of word counts. In each dictionary, each term would be a key, and its count would be the value.

Within this problem, we have to think of two scenarios:

For each word in our corpus,

1. If we are encountering a new word, that is, if the word is not in our data structure, create an item where the key is the token, and the value is 1.
1. Otherwise, if we are not encountering a new word, retrieve the value and update it by adding 1 to the count.

Remember how to access, create and modify information in a dictionary. To retrieve a value from a dictionary, you call the dictionary name and the key in brackets. This method also works to create a new dictionary item, or to update an item's value. Also remember that you can iterate through the keys in a dictionary as you would iterate through the items in a list.


Now, write some code that:
1. Creates an empty dictionary called corpus_counts
2. Loops through the terms in the corpus_tokens and:
     - If the term does not have an entry in corpus_counts, creates an entry with the term for the key and value 1
     - If the term does have an entry in corpus_counts, updates the entry and assigns it its value + 1.

In [ ]:
corpus_counts={}
for token in corpus_tokens:
    if token not in corpus_counts:
        corpus_counts[token]=1
    else:
        corpus_counts[token]=corpus_counts[token]+1

With the loop above, we should have created a dictionary that includes all the terms in the volume, and the number of times each term appears. But how can we access this information in a way that allows us to draw conclusions, for instance, about the most frequent words in the book?

Recall that dictionaries are unordered by nature.  Though we can sort by key, what we want to be doing is to sort by value. Because values could be repeated, and thus won't serve as unique dictionary keys, we can't just create a dictionary of value: key pairs. The solution to this dilemma is difficult, so we provide it below. What we need is an ordered collection that that gives us value, key pairs, with values first and keys second. Using a list of value, key touples, we could store repeated values and reorder for analysis.

To resolve this issue,

1. Create an empty list called ct_value_list
1. Create a for-loop that iterates through key,value pairs in the items of the corpus_counts dictionary
    1. For each key, value pair, append a tuple consisting of the (value, key) to ct_value_list.

In [ ]:
ct_value_list =[]
for k,v in corpus_counts.items():
    ct_value_list.append((v,k))

We can now sort these using the sorted() function, using the keyword reverse=True to order them from largest to smallest.

Reassign the ct_value_list variable to a sorted version of the list.

In [ ]:
ct_value_list = sorted(ct_value_list, reverse=True)

And now, we can index into the list to retrieve, for instance, the 10 most frequent words, by accessing the first ten items.

In [ ]:
ct_value_list[0:20]

One thing we notice immediately is that the most frequent words in the text don't actually give us much of an idea as to what is going on in the text.

> **Stop words** are words in a language that have very little meaning and are filtered out before or after natural language processing.

We could filter these out from our analysis because many are very frequent, and will obscure the importance of other terms that might provide more meaning.

Below, we create a list of words that are commonly considered stop words in the English language.

In [ ]:
stopwords= ['i','me','my','myself','we','our','ours','ourselves','you',"youre","youve","youll","youd",'your',
 'yours', 'yourself','yourselves','it','its','itself','they','them','their','theirs','themselves','what','which','who','whom','this',
 'that',"thatll",'these','those','am','is','are','was','were','be','been','being','have','has','had',
 'having','do','does','did','doing','a','an','the','and','but','if','or','because','as','until','while',
 'of','at','by','for','with','about','against','between','into','through','during','before','after','above',
 'below','to','from','up','down','in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then',
 'once', 'here', 'there', 'when', 'where', 'why','how', 'all','any', 'both','each', 'few', 'more', 'most',
 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't',
 'can', 'will', 'just', 'don', "dont", 'should', "shouldve", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y',
 'ain', 'aren', "arent", 'couldn', "couldnt", 'didn', "didn't", 'doesn', "doesnt", 'hadn', "hadnt",
 'hasn', "hasnt", 'haven', "havent", 'isn', "isnt", 'ma', 'mightn', "mightnt", 'mustn', "mustnt",
 'needn', "neednt", 'shan', "shant", 'shouldn', "shouldnt", 'wasn', "wasnt", 'weren', "werent", 'won',
 "wont", 'wouldn', "wouldnt",'said','thou','one','went','came','thee','could','would','took','go','shall','must','however','thy']

In [ ]:
# You can create the list by extracting the stop words from the text

stop2 = []

for stopword in stopwords:
   if stopword not in stop2 and stopword in corpus_counts:
       stop2.append(stopword)

In [ ]:
print(stop2)

Now, let's repeat the loop we made before, but with the additional condition that the token/term should only be added to the dictionary if it is NOT in our stopword list.

In [ ]:
corpus_counts={}
for token in corpus_tokens:
    if token not in stopwords:
        if token not in corpus_counts:
            corpus_counts[token]=1
        else:
            corpus_counts[token]=corpus_counts[token]+1

Now, if we order the contents of the dictionary and print them, we get a clearer idea of the contents of the volume.

In [ ]:
ct_value_list_stop =[]
for k,v in corpus_counts.items():
    ct_value_list_stop.append((v,k))

In [ ]:
ct_value_list_stop = sorted(ct_value_list_stop, reverse=True)

In [ ]:
ct_value_list_stop[0:20]

And below, we will use Pandas (a module you will learn about in Lessons 9-10) to visualize this information. We can visualize the fifty most common terms in this collection of stories:

In [ ]:
import pandas as pd

c_counts_df=pd.DataFrame.from_dict(corpus_counts,orient='index',columns=['count']).sort_values(by='count',ascending=False)
c_counts_df[0:50].sort_values(by='count',ascending=False).plot.bar(figsize=(17,5))

One thing that stands out is that words denoting male characters (he, king, man, father,  kings, son, hans) are overall more frequent than those relating to females (woman, daughter, wife). What is more, it seems that men are more likely to be addressed as independent agents than to be mentioned in relation to somebody else. This exploratory analysis might lead to a research question: are men and women described differently in Grimms Fairy Tales?

We have gone ahead and made a list of common words denoting male or female characters from our counts above.

In [ ]:
men = ['king', 'kings', 'man', 'father', 'son', 'sons', 'brother', 'master', 'huntsman', 'boy', 'lord',
       'husband', 'soldier', 'devil', 'men', 'prince', 'dwarf', 'huntsmen', 'bridegroom', 'sons', 'fathers',
       'soldiers', 'godfather', 'boys', 'dwarfs', 'captain', 'shepherd', 'ferdinand', 'farmer',
       'countryman','his','him','he','himself']

women = ['woman', 'daughter', 'daughters', 'wife', 'mother', 'maiden', 'girl', 'queen', 'bride', 'princess',
         'sister', 'witch', 'stepmother', 'sisters', 'daughters', 'grandmother', 'waitingmaid','cinderella',
         'rapunzel','her','hers','she','herself']

The first thing we could ask is: are men mentioned more than women?

In [ ]:
mentions = 0
for token in men:
    mentions= mentions + corpus_counts[token]

In [ ]:
mentions

In [ ]:
womentions = 0
for token in women:
    womentions= womentions + corpus_counts[token]

In [ ]:
womentions

This seems like a good starting point for a more fine-grained analysis of the corpus of stories in Grimms' Fairy Tales.

# 3. Very simple gender analysis

Before, we counted the terms in our corpus as a whole. We can fine-tune our analysis by changing the level of aggretation, looking at counts not within the corpus, but within stories individually.


## Splitting the block of text into stories
A first step would be to split the block of text into stories. Previously, we observed that the stories in the text are separated by four newline characters (\n). Use this information and a [string method](https://docs.python.org/2.5/lib/string-methods.html) to split the clean text into a list of stories, assigned to a variable called 'stories'.

In [ ]:
stories = text_clean.split('\n\n\n\n')

How many stories do we have? Check the length of the list, which should be 201.

In [ ]:
len(stories)

Finally, retrieve and print an item from the list, to examine its contents:

In [ ]:
print(stories[40])

As we did with the whole corpus, we will have to  tokenize each individual story.

In the code cell below, create an empty list called stories_tokens. Then build a loop that iterates through each item in our stories_dict, replaces all \n characters with spaces, splits each cleaned story by spaces, and appends the results to the stories_tokens list.

In [ ]:
stories_tokens = []
for story in stories:
    stories_tokens.append(story.replace('\n',' ').split(' '))

Where we started with a list of stories, we further split each of those items into a list of terms. It is useful to thus bear in mind that stories_tokens is a list of lists. Each item in stories_tokens, indexable by position, is a list of the terms included in a particular story.  

In this situation, we can access a story by position, which has been preserved in the previous transformation from individual poem to list of tokens, is sufficient for our purposes.

Let's examine one item from our new poems_tokens list. What it should contain is a list of each of the terms in our poem.

In [ ]:
print(stories_tokens[40])

By now, we have the raw data in a format that allows us to begin our analysis.

Next, we will count terms within a story,  to evaluate term frequencies, which we can do by counting terms within each document and dividing by the document's length. Once we have this information, we will evaluate document similarity by creating a formula that compares term frequencies in two documents.

# Calculating male/female frequencies per story

For a more in-depth analysis could count all terms again, but per story. Since we are only interested in a very basic measure of the number of times a male or female character is mentioned, we can construct a loop for just this purpose. But instead of counting the terms in the text overall, it should loop through the stories_tokens list and count the terms in each story.

Create an empty list called stories_mentions.
Create a loop that iterates through each story in the stories_tokens list. Have it creates a stories_length variable for the length of the story, and two variables mentions and womentions equal to 0.
Then construct a nested for-loop that iterates through the tokens in the story, adding 1 to the value of mentions and womentions if a token is in the male or female lists, respectively.  
Finally, append an item to the stories_mentions list that is a list with the mentions, womentions and stories_length as an item.

In [ ]:
stories_mentions = []

for story in stories_tokens:
    story_length=len(story)
    mentions= 0
    womentions =0
    for token in story:
        if token in men:
            mentions = mentions+1
        elif token in women:
            womentions = womentions+1
    mentions=mentions
    womentions= womentions
    stories_mentions.append([mentions,womentions, story_length])

In [ ]:
stories_mentions

# Visualizing some questions

We can now bring this information into Pandas and visualize some patterns. You will learn about Pandas in lessons 9 and 10- for now, this is an efficient way to analyze and display the data you gathered.

First, we structure our data (you do not need to be able to understand this yet).

In [ ]:
stories_mentions_df = pd.DataFrame(stories_mentions,columns=['m_mentions','f_mentions','length'])

Below, we calculate frequency of male/female mentions by dividing the counts by story length. This allows us to control for the effects of having very long or short stories.

We also create a variable for the difference between the frequencies, which allows us to see whether a story skews male or female.

In [ ]:
stories_mentions_df['f_freq']= stories_mentions_df['f_mentions']/stories_mentions_df['length']
stories_mentions_df['m_freq']= stories_mentions_df['m_mentions']/stories_mentions_df['length']
stories_mentions_df['difference']= stories_mentions_df['f_freq']-stories_mentions_df['m_freq']

We add a variable called 'gender', with values 'male', 'female' or 'mixed', to indicate whether there are only male or female mentions, or a combination of both, in the story.

In [ ]:
# Add gender variable
stories_mentions_df['gender']='mixed'
stories_mentions_df.loc[(stories_mentions_df['m_mentions']>0)&(stories_mentions_df['f_mentions']==0),'gender']='male'
stories_mentions_df.loc[(stories_mentions_df['f_mentions']>0)&(stories_mentions_df['m_mentions']==0),'gender']='female'

And we add the story titles to our data.

In [ ]:
# Add titles
story_titles=[]
with open(os.path.join(f"{module_path}/Other_files/", "GrimmsTitles.txt")) as file:
    for line in file:
        story_titles.append(line.strip())
for index, item in enumerate(story_titles):
    stories_mentions_df.loc[index,'title']=item

In [ ]:
stories_mentions_df

In [ ]:
# Import modules to help visualize
import numpy as np
import matplotlib.pyplot as plt

### Visualization 1: number of stories with only male or only female mentions

*   List item
*   List item



This first visualization is a simple bar chart that counts the number of stories with mixed, only male or only female mentions. The majority of stories (160) mention characters of each gender. 30 stories have only male characters. And 3 have only female characters.

In [ ]:
stories_mentions_df['gender'].value_counts().plot(kind='bar',figsize=(10,5))

### Visualization 2: Histogram of the Difference between Male and Female Mentions

Below, we plot a histogram of the 'difference' variable for stories with both types of characters. A histogram is a graphical representation of how a variable is distributed. It groups a variable's values into intervals or bins, and counts how many values fall into each bin.

In this graph, the difference is more or less distributed as a bell curve (perhaps with a bit of a right tail). But as can be seen, the center of the bell curve is not found at the value 0 (neutral story), but somewhere in the negative (blue, or male) side. We can  thus see that the difference (calculated as f_mentions - m_mentions) is more often negative than positive (below zero, or in blue), meaning that stories tend to mention male characters more than female characters.

We have excluded stories that only mention males or females.

In [ ]:
cm = plt.cm.get_cmap('RdYlBu_r')

Y,X = np.histogram(stories_mentions_df.loc[stories_mentions_df['gender']=='mixed','difference'], 20)
x_span = X.max()-X.min()
C = [cm(((x-X.min())/x_span)) for x in X]

plt.bar(X[:-1],Y,color=C,width=X[1]-X[0])
plt.show()

In [ ]:
colors = {'male':'red', 'female':'blue', 'mixed':'purple'}

# Visualization 3: area chart of male and female frequencies, plotted over the duration of the book

The chart below shows frequency of male and female mentions (the female mentions plotted on a negative axis, so they can be viewed separately) per story, with the stories ordered as they appear in the book. We can see that male mentions tend to predominate, but at a first glance there does not seem to be much of a pattern in the ordering of the stories, at least regarding male- or female-dominated narratives.

In [ ]:
stories_mentions_df['neg_f_freq']=-1*stories_mentions_df['f_freq']
ax = stories_mentions_df.plot.area('title',['m_freq','neg_f_freq'],figsize=(15,5))
ax.tick_params(axis='x', rotation=45)

With just a simple bit of data gathering, we were able to perform some analysis that might spark further questions. To go further, we could move beyond mentions and look at the number of characters of each gender mentioned in a story. Or we could look at how male and female characters are described, either by the adjectives surrounding them or in the way their actions are described.  We could also go beyond the contents of the book itself and think about reception: of the Grimms' fairy tales, some are more famous than others. How do these reflect the male and female characters they contain?